In [ ]:
from _common import *

## Notebook 05 -- MLP

### Section 1 -- MLflow tracking, imports, budget constants

In [ ]:
import mlflow

MLFLOW_DB_PATH = ROOT_FOLDER / 'notebooks' / 'mlflow_mlp.db'
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB_PATH}")
mlflow.set_experiment("05_mlp")

In [ ]:
import optuna
from optuna.samplers import TPESampler
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import json
import time

optuna.logging.set_verbosity(optuna.logging.WARNING)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)

In [ ]:
MAX_TRIALS = 100
PATIENCE = 15

### Section 2 -- MLPTorch, MLPRegressor wrapper, make_model(), search space

**Design note, matching `04_xgboost.ipynb`'s decision (plan doc SS4b).** `n_epochs`
is tuned as a plain Optuna hyperparameter, not chosen by internal early stopping on
a held-out slice. This keeps `inner_cv_score()` / `inner_cv_score_subject()`
unchanged -- every baseline model's `.fit(X, y)` takes exactly a training matrix and
a label vector, nothing else -- so the three baseline notebooks stay copy-paste
siblings (plan doc decision 5) differing only in `make_model()` and the search
space, per the fairness contract.

`MLPRegressor` below is a small sklearn-duck-typed wrapper (`.fit` / `.predict`
only, no `BaseEstimator` inheritance needed since nothing here calls `clone()`) so
it slots into a `Pipeline` exactly like Ridge's `Pipeline([StandardScaler, Ridge])`
-- an MLP is scale-sensitive the same way Ridge is (protocol doc SS3), unlike the
tree models.

Architecture is a plain feed-forward stack, width halving each layer
(`hidden_dims_from`), 1-3 hidden layers -- kept small on purpose per
[[feedback_xai_scope_control]] and because 11x9 markers is not a large input by deep
-learning standards; this notebook's job (protocol doc SS1) is only to isolate
"neural vs. tree" on the same feature vector, not to chase depth.

Batch size is tuned, not fixed -- real-data timing (fold 0, rung 1 vs. rung 3 scale)
showed batch=4096 runs a rung-3-scale epoch (~272k rows) in ~0.19s vs. ~0.54s at
batch=1024, so batch size is a genuine cost lever here, unlike Ridge/XGBoost where
it does not apply. Search stays within {1024, 2048, 4096, 8192} -- large enough to
matter, small enough that gradient noise still varies across trials.

In [ ]:
class MLPTorch(nn.Module):
    def __init__(self, n_features, hidden_dims, dropout):
        super().__init__()
        layers = []
        prev = n_features
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers += [nn.Linear(prev, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class MLPRegressor:
    """sklearn-duck-typed wrapper: .fit(X, y) / .predict(X). n_epochs is a tuned
    hyperparameter (see design note above), not data-determined -- a single .fit
    call trains for exactly n_epochs, no internal validation split."""

    def __init__(self, hidden_dims, dropout, lr, weight_decay, batch_size, n_epochs, random_state=RANDOM_SEED):
        self.hidden_dims = hidden_dims
        self.dropout = dropout
        self.lr = lr
        self.weight_decay = weight_decay
        self.batch_size = batch_size
        self.n_epochs = n_epochs
        self.random_state = random_state

    def fit(self, X, y):
        torch.manual_seed(self.random_state)
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32)
        self.model_ = MLPTorch(X.shape[1], self.hidden_dims, self.dropout).to(DEVICE)
        opt = torch.optim.Adam(self.model_.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        loss_fn = nn.MSELoss()

        X_t = torch.from_numpy(X).to(DEVICE)
        y_t = torch.from_numpy(y).to(DEVICE)
        n = len(X_t)

        self.model_.train()
        for epoch in range(self.n_epochs):
            perm = torch.randperm(n, device=DEVICE)
            for start in range(0, n, self.batch_size):
                idx = perm[start:start + self.batch_size]
                opt.zero_grad()
                pred = self.model_(X_t[idx])
                loss = loss_fn(pred, y_t[idx])
                loss.backward()
                opt.step()
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=np.float32)
        X_t = torch.from_numpy(X).to(DEVICE)
        self.model_.eval()
        with torch.no_grad():
            pred = self.model_(X_t)
        return pred.cpu().numpy()


def hidden_dims_from(width, n_layers):
    dims = []
    w = width
    for _ in range(n_layers):
        dims.append(max(w, 8))
        w = max(w // 2, 8)
    return dims


def make_model(params):
    hidden_dims = hidden_dims_from(params['width'], params['n_layers'])
    return Pipeline([
        ('scale', StandardScaler()),
        ('mlp', MLPRegressor(
            hidden_dims=hidden_dims,
            dropout=params['dropout'],
            lr=params['lr'],
            weight_decay=params['weight_decay'],
            batch_size=params['batch_size'],
            n_epochs=params['n_epochs'],
            random_state=RANDOM_SEED,
        )),
    ])


def make_objective(fold):
    def objective(trial):
        t0 = time.time()
        params = {
            'width': trial.suggest_int('width', 32, 512, log=True),
            'n_layers': trial.suggest_int('n_layers', 1, 3),
            'dropout': trial.suggest_float('dropout', 0.0, 0.5),
            'lr': trial.suggest_float('lr', 1e-4, 1e-2, log=True),
            'weight_decay': trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True),
            'batch_size': trial.suggest_categorical('batch_size', [1024, 2048, 4096, 8192]),
            'n_epochs': trial.suggest_int('n_epochs', 20, 150),
        }
        score = inner_cv_score(fold, params, make_model)
        print(f"  trial {trial.number}: {time.time() - t0:.1f}s, n_epochs={params['n_epochs']}, width={params['width']}, score={score:.3f}")
        return score
    return objective


def stop_when_flat(study, trial):
    # study-level patience -- the only stopping rule (Optuna pruning is OFF, see
    # plan doc traps). Stop once PATIENCE trials have passed with no new best.
    if trial.number - study.best_trial.number >= PATIENCE:
        study.stop()

### Rung 1 -- nested tuning, all session folds

In [ ]:
def _json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): _json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    return obj


def _save_checkpoint(results):
    with open(FOLD_METRICS_PATH, 'w') as f:
        json.dump(_json_safe(results), f, indent=2)


FOLD_METRICS_PATH = ROOT_FOLDER / 'notebooks' / 'fold_metrics_mlp.json'

if FOLD_METRICS_PATH.exists():
    with open(FOLD_METRICS_PATH) as f:
        fold_results = json.load(f)
else:
    fold_results = []

done_keys = {(r['participant_id'], r['test_session']) for r in fold_results}
print(f"resuming with {len(fold_results)} folds already done, {len(SESSION_FOLDS) - len(fold_results)} remaining")

In [ ]:
study_first_new = None   # the first fold actually run THIS session -- used for the
                          # convergence diagnostic below, not for scoring

for fold in SESSION_FOLDS:
    p_num = fold['participant_id']
    test_session = fold['test_session']

    if (p_num, test_session) in done_keys:
        continue

    sampler = TPESampler(seed=RANDOM_SEED)
    study = optuna.create_study(direction='minimize', sampler=sampler)
    study.optimize(make_objective(fold), n_trials=MAX_TRIALS, callbacks=[stop_when_flat])

    if study_first_new is None:
        study_first_new = study

    best_params = dict(study.best_params)

    train_mask = (FEATURES['participant_id'] == p_num) & (FEATURES['session_id'].isin(fold['train_sessions']))
    test_mask = (FEATURES['participant_id'] == p_num) & (FEATURES['session_id'] == test_session)

    # leak guard -- the outer test session must never appear in the training rows
    assert not (FEATURES.loc[train_mask, 'session_id'] == test_session).any()
    assert set(FEATURES.loc[train_mask, 'session_id'].unique()) == set(fold['train_sessions'])

    X_train = FEATURES.loc[train_mask, FEATURE_COLS]
    y_train = FEATURES.loc[train_mask, 'pct_mvc_corr']
    X_test = FEATURES.loc[test_mask, FEATURE_COLS]
    y_test = FEATURES.loc[test_mask, 'pct_mvc_corr']

    final_model = make_model(best_params)
    final_model.fit(X_train, y_train)
    y_pred = final_model.predict(X_test)

    result = evaluate(y_test, y_pred, FEATURES.loc[test_mask, 'phase'], FEATURES.loc[test_mask, 'target_pct'])

    with mlflow.start_run(run_name=f"p{p_num}_s{test_session}"):
        mlflow.log_params(best_params)
        mlflow.log_metric('rmse', result['overall']['rmse'])
        mlflow.log_metric('mae', result['overall']['mae'])
        mlflow.log_metric('r2', result['overall']['r2'])
        mlflow.log_metric('n_trials', len(study.trials))

    fold_results.append({
        'participant_id': p_num,
        'fold': fold['fold'],
        'test_session': test_session,
        'best_params': best_params,
        'n_trials': len(study.trials),
        'metrics': result,
    })
    done_keys.add((p_num, test_session))
    _save_checkpoint(fold_results)

    print(p_num, test_session, len(study.trials), result['overall'])

print(f"done -- {len(fold_results)} / {len(SESSION_FOLDS)} folds in fold_metrics_mlp.json")

### Convergence check -- one fold only

In [ ]:
from optuna.visualization import plot_optimization_history

if study_first_new is not None:
    plot_optimization_history(study_first_new)
else:
    print("nothing new ran this session -- no study to plot")

### Rung 3 -- cross-subject, cold

Held-out unit is a whole **participant**, not a session -- every session they have is
excluded from training, and there is no fine-tuning (that is rung 4). Uses
`SUBJECT_FOLDS` (8 folds, one per participant) and `INNER_VAL_PARTICIPANTS`
(`_common.py` SS2b) -- the same fixed 3-participant round-robin used by
`03_ridge.ipynb` and `04_xgboost.ipynb`'s rung 3. Tuning goes through
`inner_cv_score_subject()` (`_common.py` SS6b) instead of `inner_cv_score()`.

Same search space, same `MAX_TRIALS`/`PATIENCE`, same `make_model()` as rung 1 --
only the fold source and the inner-CV function differ. Writes
`fold_metrics_mlp_rung3.json` (separate checkpoint file, keyed by `participant_id`
not `(participant_id, test_session)`). Logs to the same `mlflow_mlp.db` /
`05_mlp` experiment as rung 1, with a distinct run-name scheme (`rung3_p{p}` vs
rung 1's `p{p}_s{s}`) so the two rungs' runs never collide.

**Per the same gate rung 1's siblings used (Ridge SS4d, XGBoost SS4c/SS4b): time
fold 0 before committing to the full budget.** Real-data timing done ahead of this
build (not a notebook cell, script-based like Ridge's rung-3 SS4d note) found a
*warm* MLP epoch costs ~0.04s at rung-1 scale (~24k rows/fold) and ~0.19s at
rung-3 scale (~272k rows/fold, batch=4096) -- cheap per epoch, so unlike XGBoost
the per-trial cost here is not dominated by epoch count. The timing check below
re-confirms this inside the real notebook/kernel before the full loop runs.

In [ ]:
def make_objective_subject(fold):
    def objective(trial):
        t0 = time.time()
        params = {
            'width': trial.suggest_int('width', 32, 512, log=True),
            'n_layers': trial.suggest_int('n_layers', 1, 3),
            'dropout': trial.suggest_float('dropout', 0.0, 0.5),
            'lr': trial.suggest_float('lr', 1e-4, 1e-2, log=True),
            'weight_decay': trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True),
            'batch_size': trial.suggest_categorical('batch_size', [1024, 2048, 4096, 8192]),
            'n_epochs': trial.suggest_int('n_epochs', 20, 150),
        }
        score = inner_cv_score_subject(fold, INNER_VAL_PARTICIPANTS[fold['fold']], params, make_model)
        print(f"  trial {trial.number}: {time.time() - t0:.1f}s, n_epochs={params['n_epochs']}, width={params['width']}, score={score:.3f}")
        return score
    return objective

# make_model() and stop_when_flat() are reused unchanged from rung 1 (cell above).

#### Fold-0 timing check -- confirm the budget before the full 8-fold run

5 trials, no patience callback -- same smoke-test shape used by 03/04's rung 3.
Not a real tuning run; purely a per-trial wall-clock measurement to extrapolate
from before spending real compute on all 8 folds.

In [ ]:
_t0 = time.time()
timing_sampler = TPESampler(seed=RANDOM_SEED)
timing_study = optuna.create_study(direction='minimize', sampler=timing_sampler)
timing_study.optimize(make_objective_subject(SUBJECT_FOLDS[0]), n_trials=5)
_t1 = time.time()

per_trial = (_t1 - _t0) / 5
projected_fold_minutes = per_trial * MAX_TRIALS / 60
projected_total_hours = per_trial * MAX_TRIALS * len(SUBJECT_FOLDS) / 3600

print(f"rung 3 fold 0, 5 trials, no patience: {per_trial:.2f} s/trial")
print(f"worst case (no patience firing) at MAX_TRIALS={MAX_TRIALS}: "
      f"{projected_fold_minutes:.1f} min/fold, {projected_total_hours:.1f} h for all {len(SUBJECT_FOLDS)} folds")

#### Rung 3 -- nested tuning, all 8 folds

**Review the timing check above before running this cell.** If the projected total
does not survive (same failure mode as XGBoost's rung 1, plan doc SS4c), lower
`MAX_TRIALS`/`PATIENCE` for rung 3 specifically rather than assuming rung 1's
numbers transfer -- do not silently shrink the search space to compensate without
flagging it.

**Resumable**, same pattern as rung 1: loads `fold_metrics_mlp_rung3.json` if
present, skips any `participant_id` already in it, saves after every new fold.

In [ ]:
FOLD_METRICS_RUNG3_PATH = ROOT_FOLDER / 'notebooks' / 'fold_metrics_mlp_rung3.json'

if FOLD_METRICS_RUNG3_PATH.exists():
    with open(FOLD_METRICS_RUNG3_PATH) as f:
        fold_results_rung3 = json.load(f)
else:
    fold_results_rung3 = []

done_keys_rung3 = {r['participant_id'] for r in fold_results_rung3}
print(f"resuming with {len(fold_results_rung3)} folds already done, {len(SUBJECT_FOLDS) - len(fold_results_rung3)} remaining")


def _save_checkpoint_rung3(results):
    with open(FOLD_METRICS_RUNG3_PATH, 'w') as f:
        json.dump(_json_safe(results), f, indent=2)


for fold in SUBJECT_FOLDS:
    test_p = int(fold['test_participant'].removeprefix('participant'))

    if test_p in done_keys_rung3:
        continue

    sampler = TPESampler(seed=RANDOM_SEED)
    study = optuna.create_study(direction='minimize', sampler=sampler)
    study.optimize(make_objective_subject(fold), n_trials=MAX_TRIALS, callbacks=[stop_when_flat])

    best_params = dict(study.best_params)

    train_participant_nums = [int(p.removeprefix('participant')) for p in fold['train_participants']]
    train_mask = FEATURES['participant_id'].isin(train_participant_nums)
    test_mask = FEATURES['participant_id'] == test_p

    # leak guard -- the outer test participant must never appear in the training rows
    assert not (FEATURES.loc[train_mask, 'participant_id'] == test_p).any()
    assert set(FEATURES.loc[train_mask, 'participant_id'].unique()) == set(train_participant_nums)

    X_train = FEATURES.loc[train_mask, FEATURE_COLS]
    y_train = FEATURES.loc[train_mask, 'pct_mvc_corr']
    X_test = FEATURES.loc[test_mask, FEATURE_COLS]
    y_test = FEATURES.loc[test_mask, 'pct_mvc_corr']

    final_model = make_model(best_params)
    final_model.fit(X_train, y_train)
    y_pred = final_model.predict(X_test)

    result = evaluate(y_test, y_pred, FEATURES.loc[test_mask, 'phase'], FEATURES.loc[test_mask, 'target_pct'])

    with mlflow.start_run(run_name=f"rung3_p{test_p}"):
        mlflow.log_params(best_params)
        mlflow.log_metric('rmse', result['overall']['rmse'])
        mlflow.log_metric('mae', result['overall']['mae'])
        mlflow.log_metric('r2', result['overall']['r2'])
        mlflow.log_metric('n_trials', len(study.trials))

    fold_results_rung3.append({
        'participant_id': test_p,
        'fold': fold['fold'],
        'best_params': best_params,
        'n_trials': len(study.trials),
        'metrics': result,
    })
    done_keys_rung3.add(test_p)
    _save_checkpoint_rung3(fold_results_rung3)

    print(test_p, len(study.trials), result['overall'])

print(f"done -- {len(fold_results_rung3)} / {len(SUBJECT_FOLDS)} folds in fold_metrics_mlp_rung3.json")

In [ ]:
rmses_rung3 = np.array([r['metrics']['overall']['rmse'] for r in fold_results_rung3])

print(f"MLP, rung 3, {len(rmses_rung3)} folds")
print(f"mean RMSE {rmses_rung3.mean():.2f} %MVC (SD {rmses_rung3.std(ddof=1):.2f}, "
      f"range {rmses_rung3.min():.2f}-{rmses_rung3.max():.2f})")

### Summary

In [ ]:
rmses = np.array([r['metrics']['overall']['rmse'] for r in fold_results])

print(f"MLP, rung 1, {len(rmses)} folds")
print(f"mean RMSE {rmses.mean():.2f} %MVC (SD {rmses.std(ddof=1):.2f}, range {rmses.min():.2f}-{rmses.max():.2f})")